## 09 — Published Panel Comparison

Evaluate published AISNP panels (Shi 2019, Cao 2022, Cai 2024) on the same
504-sample CN/JPT/SEA task using identical 5-fold CV pipeline as Stage 2 of `08`.

Also checks SNP overlap between PAANDA-EA committed panels and each published panel.

In [1]:
import os, sys, json, subprocess
import urllib.request
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Locate project root dynamically
_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts" / "config.py").exists())
sys.path.insert(0, str(_root / "scripts"))
from notebook_init import setup
_cfg, PATHS, POPULATIONS, HARD_FILTERS, SITUATIONAL_FILTERS, ML = setup()

PANELS_DIR  = _root / "data" / "published_panels"
OUTPUT_DIR  = PATHS.outputs_dir("self_evaluation/09_published_panel_comparison")
OUR_PANELS  = PATHS.outputs_dir("self_evaluation/08_unified_panel_sweep") / "panels"
MAF_PLINK   = str(PATHS.outputs_dir("01_hard_filtering") / "SEA_JPT_CN_MAF_filtered")
PSAM        = str(PATHS.outputs_dir("02_situational_filtering") / "SEA_JPT_CN_LD_pruned.psam")
GENO_RAW    = str(OUTPUT_DIR / "published_panels_geno.raw")
COORD_CACHE = str(OUTPUT_DIR / "rsid_coords.json")

os.makedirs(str(OUTPUT_DIR), exist_ok=True)
print(f"Output: {OUTPUT_DIR}")
print(f"Panels: {PANELS_DIR}")


Project root : /home/ibmelab/Projects/AISNP_Research
genomes_data : /mnt/data/aisnp_data/1000genomes/
output root  : output
Output: /mnt/data/aisnp_data/1000genomes/outputs/self_evaluation/09_published_panel_comparison
Panels: /home/ibmelab/Projects/AISNP_Research/data/published_panels


In [2]:
# Population labels from PSAM  (format: #IID  pop)
pop_labels = {}
with open(PSAM) as f:
    for line in f:
        if line.startswith("#"): continue
        parts = line.strip().split("	")
        iid, pop = parts[0], parts[1]   # IID=col0, pop=col1
        pop_labels[iid] = pop

print(f"Samples with labels: {len(pop_labels)}")
print("Populations:", {v: list(pop_labels.values()).count(v) for v in set(pop_labels.values())})


Samples with labels: 504
Populations: {'CN': 208, 'JPT': 104, 'SEA': 192}


In [3]:
# Load rsID lists for each published panel
def load_rsids(fname):
    p = PANELS_DIR / fname
    return [l.strip() for l in open(p) if l.strip()]

PANELS = {
    'shi_36':  load_rsids('shi_36.txt'),
    'shi_59':  load_rsids('shi_59.txt'),
    'shi_98':  load_rsids('shi_98.txt'),
    'shi_142': load_rsids('shi_142.txt'),
    'cao_19':  load_rsids('cao_19.txt'),
}

# cai_eas34: already has coordinates — load separately
cai_coords = []
with open(str(PANELS_DIR / 'cai_eas34_coords.tsv')) as f:
    for line in f:
        parts = line.strip().split('\t')
        cai_coords.append({'chrom': parts[0], 'pos': parts[1],
                           'rsid': parts[2], 'ref': parts[3], 'alt': parts[4]})

all_rsids = sorted({r for rs in PANELS.values() for r in rs})
print(f'Unique rsIDs to map (shi+cao): {len(all_rsids)}')
print(f'Cai EAS34 entries (with coords): {len(cai_coords)}')


Unique rsIDs to map (shi+cao): 161
Cai EAS34 entries (with coords): 34


In [4]:
# Query MyVariant.info for chr:pos (b37) — cached
if os.path.exists(COORD_CACHE):
    with open(COORD_CACHE) as f:
        coord_map = json.load(f)
    print(f'Loaded coord cache: {len(coord_map)} entries')
else:
    print('Querying MyVariant.info...')
    data = json.dumps({
        'q': all_rsids, 'scopes': 'dbsnp.rsid',
        'fields': 'dbsnp.rsid,dbsnp.chrom,dbsnp.hg19.start,dbsnp.ref,dbsnp.alt',
        'size': 1
    }).encode()
    req = urllib.request.Request(
        'https://myvariant.info/v1/query', data=data,
        headers={'Content-Type': 'application/json', 'User-Agent': 'python'})
    results = json.loads(urllib.request.urlopen(req, timeout=60).read())

    coord_map = {}
    for r in results:
        rsid = r.get('query')
        if r.get('notfound') or 'dbsnp' not in r: continue
        db = r['dbsnp']
        coord_map[rsid] = {
            'chrom': str(db.get('chrom','')),
            'pos':   str(db.get('hg19',{}).get('start','')),
            'ref':   db.get('ref',''),
            'alt':   db.get('alt',''),
        }

    # Add cai coords
    for c in cai_coords:
        coord_map[c['rsid']] = {'chrom': c['chrom'], 'pos': c['pos'],
                                'ref': c['ref'], 'alt': c['alt']}

    with open(COORD_CACHE, 'w') as f:
        json.dump(coord_map, f)
    print(f'Mapped {len(coord_map)} rsIDs, cached.')


Loaded coord cache: 182 entries


In [5]:
# Search MAF-filtered pvar for matching positions, extract genotypes (cached)
if os.path.exists(GENO_RAW):
    print(f'Genotype cache exists: {GENO_RAW}')
else:
    print('Searching pvar for published panel positions...')
    pos_set = {(c['chrom'], c['pos']) for c in coord_map.values()}

    # Write positions file for awk
    pos_file = str(OUTPUT_DIR / 'target_positions.tsv')
    with open(pos_file, 'w') as f:
        for chrom, pos in sorted(pos_set):
            f.write(f'{chrom}\t{pos}\n')

    # Find matching pvar lines
    pvar_path = MAF_PLINK + '.pvar'
    matched_ids_file = str(OUTPUT_DIR / 'matched_snp_ids.txt')
    cmd = (f"awk 'NR==FNR{{pos[$1\"_\"$2]=1;next}} /^#/{{next}} "
           f"{{key=$1\"_\"$2; if(key in pos) print $3}}' "
           f"{pos_file} {pvar_path} > {matched_ids_file}")
    subprocess.run(cmd, shell=True, check=True)
    n_matched = sum(1 for _ in open(matched_ids_file))
    print(f'Matched {n_matched} variants in pvar')

    # Extract with plink2
    out_prefix = str(OUTPUT_DIR / 'published_panels_geno')
    result = subprocess.run([
        'plink2', '--pfile', MAF_PLINK, '--keep', PSAM,
        '--extract', matched_ids_file,
        '--export', 'A', '--out', out_prefix,
        '--threads', '8', '--memory', '8000'
    ], capture_output=True, text=True)
    print(result.stdout[-500:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-300:])
    else:
        print(f'Genotype file: {GENO_RAW}')


Genotype cache exists: /mnt/data/aisnp_data/1000genomes/outputs/self_evaluation/09_published_panel_comparison/published_panels_geno.raw


In [6]:
# Parse .raw file into sample × SNP matrix
print('Parsing genotype matrix...')
with open(GENO_RAW) as f:
    header = f.readline().strip().split()
    snp_cols_raw = header[6:]   # FID IID PAT MAT SEX PHENOTYPE then SNPs

    sample_ids = []
    geno_rows  = []
    for line in f:
        parts = line.strip().split()
        iid = parts[1]
        sample_ids.append(iid)
        geno_rows.append([0 if v == 'NA' else int(float(v)) for v in parts[6:]])

G_pub = np.array(geno_rows, dtype=np.float32)
print(f'Genotype matrix: {G_pub.shape}  (samples × SNPs)')

# Population labels aligned to sample order
y_str = np.array([pop_labels[s] for s in sample_ids])
le    = LabelEncoder()
y     = le.fit_transform(y_str)
print(f'Classes: {le.classes_}  counts: {dict(zip(le.classes_, np.bincount(y)))}')

# Build col name → index in G_pub
# raw col names are like "1:40084488:\G:\T_T" → strip counted-allele suffix
col_to_idx = {}
for i, col in enumerate(snp_cols_raw):
    # plink appends _ALLELE, strip it: last underscore onward
    base = col[:col.rfind('_')]
    col_to_idx[base] = i

print(f'Column map entries: {len(col_to_idx)}')


Parsing genotype matrix...
Genotype matrix: (504, 182)  (samples × SNPs)
Classes: ['CN' 'JPT' 'SEA']  counts: {np.str_('CN'): np.int64(208), np.str_('JPT'): np.int64(104), np.str_('SEA'): np.int64(192)}
Column map entries: 182


In [7]:
# Map each published panel rsID → column index in G_pub
# plink ID format: chr:pos:\REF:\ALT  (backslashes)
def rsids_to_idx(rsid_list):
    idxs = []
    for rsid in rsid_list:
        c = coord_map.get(rsid)
        if not c: continue
        pid_fwd = f"{c['chrom']}:{c['pos']}:\\{c['ref']}:\\{c['alt']}"
        pid_rev = f"{c['chrom']}:{c['pos']}:\\{c['alt']}:\\{c['ref']}"
        if pid_fwd in col_to_idx:
            idxs.append(col_to_idx[pid_fwd])
        elif pid_rev in col_to_idx:
            idxs.append(col_to_idx[pid_rev])
    return idxs

panel_idx = {}
for pname, rsid_list in PANELS.items():
    idx = rsids_to_idx(rsid_list)
    panel_idx[pname] = idx
    print(f'  {pname}: {len(idx)}/{len(rsid_list)} SNPs matched')

# cai_eas34 via coords directly
cai_idx = []
for c in cai_coords:
    pid_fwd = f"{c['chrom']}:{c['pos']}:\\{c['ref']}:\\{c['alt']}"
    pid_rev = f"{c['chrom']}:{c['pos']}:\\{c['alt']}:\\{c['ref']}"
    if pid_fwd in col_to_idx: cai_idx.append(col_to_idx[pid_fwd])
    elif pid_rev in col_to_idx: cai_idx.append(col_to_idx[pid_rev])
panel_idx['cai_eas34'] = cai_idx
print(f'  cai_eas34: {len(cai_idx)}/{len(cai_coords)} SNPs matched')


  shi_36: 29/36 SNPs matched
  shi_59: 49/59 SNPs matched
  shi_98: 80/98 SNPs matched
  shi_142: 116/142 SNPs matched
  cao_19: 14/19 SNPs matched
  cai_eas34: 34/34 SNPs matched


In [8]:
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
import pandas as pd, pickle

CLASSIFIERS = {
    'RF':      (RandomForestClassifier(n_estimators=200, random_state=42),        False),
    'XGB':     (XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              subsample=0.8, random_state=42, verbosity=0),       False),
    'LR':      (LogisticRegression(max_iter=2000, solver='saga', random_state=42),True),
    'SVM_RBF': (SVC(kernel='rbf',    probability=True, random_state=42),          True),
    'SVM_Lin': (SVC(kernel='linear', probability=True, random_state=42),          True),
    'GBM':     (GradientBoostingClassifier(random_state=42),                      False),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def eval_panel(X_panel, y):
    records, oofpreds = [], {}
    for clf_name, (clf_tmpl, needs_scale) in CLASSIFIERS.items():
        fold_acc, fold_f1, fold_mcc, fold_auc = [], [], [], []
        oof_true = np.empty(len(y), dtype=int)
        oof_pred = np.empty(len(y), dtype=int)
        for tr_idx, te_idx in skf.split(X_panel, y):
            X_tr, X_te = X_panel[tr_idx], X_panel[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]
            if needs_scale:
                sc = StandardScaler().fit(X_tr)
                X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)
            clf = clone(clf_tmpl).fit(X_tr, y_tr)
            y_pred = clf.predict(X_te)
            oof_true[te_idx] = y_te
            oof_pred[te_idx] = y_pred
            fold_acc.append(accuracy_score(y_te, y_pred))
            fold_f1.append(f1_score(y_te, y_pred, average='weighted'))
            fold_mcc.append(matthews_corrcoef(y_te, y_pred))
            if hasattr(clf, 'predict_proba'):
                fold_auc.append(roc_auc_score(y_te, clf.predict_proba(X_te),
                                              multi_class='ovr', average='macro'))
        oofpreds[clf_name] = (oof_true.copy(), oof_pred.copy())
        records.append({'classifier': clf_name,
                        'acc':     np.mean(fold_acc),  'f1':  np.mean(fold_f1),
                        'mcc':     np.mean(fold_mcc),
                        'roc_auc': np.mean(fold_auc) if fold_auc else None})
    return records, oofpreds

all_records    = []
panel_oofpreds = {}   # panel_name → {clf: (y_true_oof, y_pred_oof)}

for pname, idxs in panel_idx.items():
    if not idxs:
        print(f'Skipping {pname}: no matched SNPs'); continue
    X_p = G_pub[:, idxs]
    print(f'\nEvaluating {pname} ({X_p.shape[1]} SNPs)...')
    recs, oofp = eval_panel(X_p, y)
    for r in recs:
        print(f"  {r['classifier']:8s}: acc={r['acc']:.4f}  f1={r['f1']:.4f}  mcc={r['mcc']:.4f}")
        all_records.append({'panel': pname, 'n_snps': X_p.shape[1], **r})
    panel_oofpreds[pname] = oofp

# Save results + OOF predictions
pd.DataFrame(all_records).to_csv(str(OUTPUT_DIR / 'published_panel_results.csv'), index=False)
with open(str(OUTPUT_DIR / 'panel_oofpreds.pkl'), 'wb') as f:
    pickle.dump({'oofpreds': panel_oofpreds, 'classes': list(le.classes_)}, f)
print(f'\nSaved results CSV and OOF predictions.')



Evaluating shi_36 (29 SNPs)...
  RF      : acc=0.6806  f1=0.6806  mcc=0.5032
  XGB     : acc=0.6587  f1=0.6587  mcc=0.4672
  LR      : acc=0.7005  f1=0.6992  mcc=0.5359
  SVM_RBF : acc=0.6806  f1=0.6755  mcc=0.5077
  SVM_Lin : acc=0.6786  f1=0.6792  mcc=0.4996
  GBM     : acc=0.6529  f1=0.6546  mcc=0.4561

Evaluating shi_59 (49 SNPs)...
  RF      : acc=0.6846  f1=0.6846  mcc=0.5090
  XGB     : acc=0.6528  f1=0.6539  mcc=0.4590
  LR      : acc=0.7084  f1=0.7093  mcc=0.5451
  SVM_RBF : acc=0.7242  f1=0.7223  mcc=0.5725
  SVM_Lin : acc=0.7004  f1=0.7007  mcc=0.5338
  GBM     : acc=0.6489  f1=0.6510  mcc=0.4498

Evaluating shi_98 (80 SNPs)...
  RF      : acc=0.7084  f1=0.7070  mcc=0.5482
  XGB     : acc=0.6926  f1=0.6912  mcc=0.5232
  LR      : acc=0.7342  f1=0.7325  mcc=0.5904
  SVM_RBF : acc=0.7758  f1=0.7744  mcc=0.6541
  SVM_Lin : acc=0.7203  f1=0.7188  mcc=0.5680
  GBM     : acc=0.6767  f1=0.6769  mcc=0.4972

Evaluating shi_142 (116 SNPs)...
  RF      : acc=0.7421  f1=0.7418  mcc=0.5

In [9]:
# Full results table per panel
print('=' * 85)
print('PUBLISHED PANEL COMPARISON — all classifiers, 5-fold CV')
print('=' * 85)
print(f'{"Panel":<12} {"N":>4}  {"Classifier":>8}  {"Acc":>7}  {"F1":>7}  {"MCC":>7}  {"ROC-AUC":>8}')
print('-' * 85)
order = ['shi_36','shi_59','shi_98','shi_142','cao_19','cai_eas34']
for pname in order:
    panel_rows = [r for r in all_records if r['panel'] == pname]
    if not panel_rows: continue
    for r in panel_rows:
        auc = f"{r['roc_auc']:.4f}" if r['roc_auc'] else '  N/A '
        print(f"{r['panel']:<12} {r['n_snps']:>4}  {r['classifier']:>8}  {r['acc']:>7.4f}"
              f"  {r['f1']:>7.4f}  {r['mcc']:>7.4f}  {auc:>8}")
    print()

# Best per panel
print('=' * 85)
print('BEST CLASSIFIER PER PANEL')
print('=' * 85)
print(f'{"Panel":<12} {"N":>4}  {"Classifier":>8}  {"Acc":>7}  {"F1":>7}  {"MCC":>7}  {"ROC-AUC":>8}')
print('-' * 85)
for pname in order:
    panel_rows = [r for r in all_records if r['panel'] == pname]
    if not panel_rows: continue
    best = max(panel_rows, key=lambda r: r['acc'])
    auc = f"{best['roc_auc']:.4f}" if best['roc_auc'] else '  N/A '
    print(f"{best['panel']:<12} {best['n_snps']:>4}  {best['classifier']:>8}  {best['acc']:>7.4f}"
          f"  {best['f1']:>7.4f}  {best['mcc']:>7.4f}  {auc:>8}")

print()
print('For reference — PAANDA-EA Stage 2 CV (stat+EN):')
print('  N=35 → acc≈0.9226   N=50 → acc≈0.9305   N=55 → acc≈0.9543')


PUBLISHED PANEL COMPARISON — all classifiers, 5-fold CV
Panel           N  Classifier      Acc       F1      MCC   ROC-AUC
-------------------------------------------------------------------------------------
shi_36         29        RF   0.6806   0.6806   0.5032    0.8268
shi_36         29       XGB   0.6587   0.6587   0.4672    0.8212
shi_36         29        LR   0.7005   0.6992   0.5359    0.8469
shi_36         29   SVM_RBF   0.6806   0.6755   0.5077    0.8364
shi_36         29   SVM_Lin   0.6786   0.6792   0.4996    0.8351
shi_36         29       GBM   0.6529   0.6546   0.4561    0.8256

shi_59         49        RF   0.6846   0.6846   0.5090    0.8344
shi_59         49       XGB   0.6528   0.6539   0.4590    0.8174
shi_59         49        LR   0.7084   0.7093   0.5451    0.8610
shi_59         49   SVM_RBF   0.7242   0.7223   0.5725    0.8560
shi_59         49   SVM_Lin   0.7004   0.7007   0.5338    0.8566
shi_59         49       GBM   0.6489   0.6510   0.4498    0.8212

shi_98   

In [10]:
import csv as _csv

# Load our panels as rsID sets using the pre-built mapping
rsid_map = {}   # sanitized snp_id → rsid
with open(str(_root / 'data' / 'published_panels' / 'our_panels_rsid_map.tsv')) as f:
    reader = _csv.DictReader(f, delimiter='\t')
    for row in reader:
        rsid_map[row['snp_id']] = row['rsid']

print(f'rsid_map loaded: {len(rsid_map)} entries, e.g. {list(rsid_map.items())[:2]}')

def load_our_panel_rsids(n):
    p = OUR_PANELS / f'panel_N{n:03d}.csv'
    if not p.exists(): return set()
    rsids = set()
    with open(str(p)) as f:
        reader = _csv.reader(f)
        next(reader)   # skip header
        for row in reader:
            snp_id = row[1]   # csv.reader handles quoted commas correctly
            rsid = rsid_map.get(snp_id)
            if rsid:
                rsids.add(rsid)
    return rsids

# Published panel rsID sets
pub_rsids = {pname: set(rsid_list) for pname, rsid_list in PANELS.items()}
pub_rsids['cai_eas34'] = {c['rsid'] for c in cai_coords}

print('SNP OVERLAP (by rsID) — PAANDA-EA committed panels vs published')
print('=' * 65)
for our_n in [35, 50, 70]:
    our_rsids = load_our_panel_rsids(our_n)
    print(f'\nPAANDA-EA N={our_n}: {len(our_rsids)} SNPs resolved to rsIDs')
    for pname, pub_set in pub_rsids.items():
        overlap = our_rsids & pub_set
        shared_str = ', '.join(sorted(overlap)) if overlap else 'none'
        print(f'  vs {pname:<12}: {len(overlap):>2} shared  [{shared_str}]')


rsid_map loaded: 70 entries, e.g. [('19:54792079_b37_G,T', 'rs431420'), ('13:95895950_b37_G,A', 'rs8003006')]
SNP OVERLAP (by rsID) — PAANDA-EA committed panels vs published

PAANDA-EA N=35: 35 SNPs resolved to rsIDs
  vs shi_36      :  0 shared  [none]
  vs shi_59      :  0 shared  [none]
  vs shi_98      :  0 shared  [none]
  vs shi_142     :  0 shared  [none]
  vs cao_19      :  2 shared  [rs2261033, rs546642722]
  vs cai_eas34   :  7 shared  [rs116783706, rs2261033, rs530357165, rs534029120, rs535319466, rs546642722, rs79200067]

PAANDA-EA N=50: 50 SNPs resolved to rsIDs
  vs shi_36      :  0 shared  [none]
  vs shi_59      :  0 shared  [none]
  vs shi_98      :  0 shared  [none]
  vs shi_142     :  0 shared  [none]
  vs cao_19      :  2 shared  [rs2261033, rs546642722]
  vs cai_eas34   :  7 shared  [rs116783706, rs2261033, rs530357165, rs534029120, rs535319466, rs546642722, rs79200067]

PAANDA-EA N=70: 70 SNPs resolved to rsIDs
  vs shi_36      :  0 shared  [none]
  vs shi_59     